In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import re
import sys
import unicodedata
import json

import numpy as np
import pandas as pd
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    import yaml
except ImportError:
    yaml = None

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

from cash_flow_forecast.contracts import (
    DatasetBuildRequest,
    GoldBuildRequest,
    SilverBuildRequest,
    build_default_schema,
)
from cash_flow_forecast.data_layers.silver import SilverBuilder
from cash_flow_forecast.data_layers.gold import GoldBuilder
from cash_flow_forecast.contracts.builders import BronzeTablePayload
from cash_flow_forecast.adapters.local import load_ruleset_from_yaml

In [ ]:
@dataclass
class SeriesConfig:
    input_path: str = "data/temp/output_clean.csv"
    ruleset_path: str = "configs/rulesets/loreal_cash_in_v1.yaml"
    date_col: str = "VALUE_DATE"
    amount_col: str = "SIGNED_AMOUNT"
    movement_col: str = "CASH_MOVEMENT_TYPE_SHORTNAME"
    movement_value: str = "TRF+"
    id_col: str = "ID"
    company_text_candidates: tuple[str, ...] = ("LABEL", "DESCRIPTION", "CPTY_NAME", "CPTY_SHORTNAME")
    company_col: str = "company_name"


cfg = SeriesConfig()

asdict(cfg)

In [ ]:
def normalize_text(value) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    text = text.upper()
    text = "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )
    text = re.sub(r"[^A-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def normalize_company(name) -> str | None:
    name = normalize_text(name)
    if not name:
        return None
    legal_forms = [
        r"\bS\s*A\s*U\b", r"\bS\s*A\b", r"\bS\s*L\s*U\b", r"\bS\s*L\b",
        r"\bB\s*V\b", r"\bN\s*V\b", r"\bS\s*A\s*R\s*L\b", r"\bS\s*C\s*A\b",
        r"\bC\s*B\b", r"\bCO\s*LTD\b", r"\bCO\s*LIMITED\b", r"\bLTD\b", r"\bLIMITED\b",
    ]
    for pattern in legal_forms:
        name = re.sub(pattern, " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    if not name or re.fullmatch(r"\d+", name):
        return None
    if re.fullmatch(r"(?:\d\s*)+", name[:9].strip()):
        return None
    if name.startswith("INCOMING"):
        return None
    return name or None


def extract_company(text) -> str | None:
    if pd.isna(text):
        return None
    text = str(text).strip()
    if not text:
        return None
    patterns = [
        r"TRANSFERENCIA DE\s+([^,]+)",
        r"/PT/FT/BO/([^/]+)",
        r"/BO/([^/]+)",
        r"^([^/]+)/",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            candidate = match.group(1).strip()
            if candidate and not re.fullmatch(r"\d+", candidate):
                return candidate
    return None


def parse_amount_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    return pd.to_numeric(
        series.astype("string")
        .str.replace("\u00A0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def first_non_empty_text(df: pd.DataFrame, columns: tuple[str, ...]) -> pd.Series:
    out = pd.Series(pd.NA, index=df.index, dtype="string")
    for col in columns:
        if col not in df.columns:
            continue
        candidate = df[col].astype("string").str.strip()
        candidate = candidate.mask(candidate.eq(""), pd.NA)
        out = out.fillna(candidate)
    return out


def sanitize_label(value: str, max_len: int = 80) -> str:
    text = normalize_text(value) or "UNKNOWN"
    text = re.sub(r"[^A-Z0-9]+", "_", text).strip("_")
    return (text or "UNKNOWN")[:max_len]

In [ ]:
def normalized_entropy(values) -> float:
    counts = pd.Series(values).value_counts(dropna=True)
    if len(counts) <= 1:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum() / np.log(len(counts)))


def top_share(values) -> float:
    counts = pd.Series(values).value_counts(dropna=True)
    if counts.sum() == 0:
        return np.nan
    return float(counts.iloc[0] / counts.sum())


def safe_cv2(values) -> float:
    arr = pd.Series(values).dropna().astype(float)
    if len(arr) <= 1:
        return np.nan
    mean = arr.mean()
    if abs(mean) < 1e-12:
        return np.nan
    return float((arr.std(ddof=1) / mean) ** 2)


def gap_stats(dates) -> dict[str, float]:
    dates = pd.Series(pd.to_datetime(dates).dropna().unique()).sort_values()
    if len(dates) <= 1:
        return {
            "median_gap_days": np.nan,
            "mean_gap_days": np.nan,
            "cv_gap_days": np.nan,
            "min_gap_days": np.nan,
            "max_gap_days": np.nan,
        }
    gaps = dates.diff().dropna().dt.days.astype(float)
    mean_gap = gaps.mean()
    cv_gap = gaps.std(ddof=1) / mean_gap if mean_gap and not np.isnan(mean_gap) else np.nan
    return {
        "median_gap_days": float(gaps.median()),
        "mean_gap_days": float(mean_gap),
        "cv_gap_days": float(cv_gap) if not np.isnan(cv_gap) else np.nan,
        "min_gap_days": float(gaps.min()),
        "max_gap_days": float(gaps.max()),
    }


def sbc_label(adi: float, cv2: float) -> str:
    """
    Syntetos-Boylan-Croston-style demand pattern label.

    Use as a diagnostic feature, not as a hard truth:
    daily treasury company series are often zero-heavy, so most companies may look intermittent/lumpy.
    """
    if pd.isna(adi) or pd.isna(cv2):
        return "unknown"
    if adi < 1.32 and cv2 < 0.49:
        return "smooth"
    if adi < 1.32 and cv2 >= 0.49:
        return "erratic"
    if adi >= 1.32 and cv2 < 0.49:
        return "intermittent"
    return "lumpy"


def longest_run(mask) -> int:
    best = current = 0
    for value in np.asarray(mask, dtype=bool):
        if value:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return int(best)


def autocorr_at_lag(values, lag: int) -> float:
    series = pd.Series(values).astype(float)
    if len(series) <= lag:
        return np.nan
    left = series.iloc[:-lag]
    right = series.iloc[lag:]
    if left.std(ddof=0) < 1e-12 or right.std(ddof=0) < 1e-12:
        return np.nan
    return float(left.corr(right))


def share_last_days(daily: pd.Series, days: int) -> float:
    total = daily.sum()
    if abs(total) < 1e-12:
        return 0.0
    cutoff = daily.index.max() - pd.Timedelta(days=days - 1)
    return float(daily.loc[daily.index >= cutoff].sum() / total)

In [ ]:
def load_ruleset_payload(path: str | Path) -> dict:
    if yaml is None:
        return {}
    ruleset_path = Path(path)
    if not ruleset_path.is_absolute():
        ruleset_path = ROOT / ruleset_path
    if not ruleset_path.exists():
        return {}
    with ruleset_path.open("r", encoding="utf-8") as handle:
        return yaml.safe_load(handle) or {}


def apply_ruleset_filters_light(df: pd.DataFrame, ruleset_payload: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    filters = ruleset_payload.get("filters", {}) if ruleset_payload else {}
    current = df.copy()
    rows = []
    for column, rule in filters.items():
        before = len(current)
        if column not in current.columns:
            rows.append({"rule": column, "status": "missing_column", "before": before, "removed": 0, "after": before})
            continue
        case_sensitive = bool(rule.get("case_sensitive", False))
        normalized = current[column].astype("string")
        if not case_sensitive:
            normalized = normalized.str.lower()
        keep = pd.Series(True, index=current.index)

        include_values = rule.get("include_values") or []
        if include_values:
            values = {str(v if case_sensitive else str(v).lower()) for v in include_values}
            keep &= normalized.isin(values)
        exclude_values = rule.get("exclude_values") or []
        if exclude_values:
            values = {str(v if case_sensitive else str(v).lower()) for v in exclude_values}
            keep &= ~normalized.isin(values)
        exclude_contains = rule.get("exclude_contains") or []
        if exclude_contains:
            values = [str(v if case_sensitive else str(v).lower()) for v in exclude_contains]
            pattern = "|".join(re.escape(v) for v in values)
            keep &= ~normalized.fillna("").str.contains(pattern, regex=True, na=False)

        current = current.loc[keep].copy()
        after = len(current)
        rows.append({"rule": column, "status": "applied", "before": before, "removed": before - after, "after": after})
    return current.reset_index(drop=True), pd.DataFrame(rows)


def canonicalize_company_names(tx: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    out = tx.copy()
    original = out["company_name"].copy()
    names = original.dropna().value_counts()
    try:
        from rapidfuzz import fuzz

        def score(a, b):
            return fuzz.token_sort_ratio(a, b)
    except ImportError:
        from difflib import SequenceMatcher

        def score(a, b):
            return 100 * SequenceMatcher(None, a, b).ratio()

    mapping = {}
    used = set()
    ordered_names = names.index.tolist()
    for name in ordered_names:
        if name in used:
            continue
        group = [name]
        used.add(name)
        for other in ordered_names:
            if other in used:
                continue
            if score(name, other) >= threshold:
                group.append(other)
                used.add(other)
        best = max(group, key=lambda n: names.loc[n])
        for item in group:
            mapping[item] = best
    manual_mapping = {
        "COFARES SDAD COOPERATIVA FARMACEUT": "COFARES SOCIEDAD COOPERATIVA FARMA",
        "SANTANDER FACTORING Y CONFIRMING E F C": "SANTANDER FACTORING Y CONFIRMING S",
        "AYESA IBERMATICA S U": "AYESA IBERMATICA",
        "PAYPAL EUROPE R L ET CIE S C A": "PAYPAL EUROPE R L ET CIE",
    }
    mapping.update(manual_mapping)
    out["company_name"] = original.map(mapping).fillna(original)
    out["company_found"] = out["company_name"].notna()
    out["company_key"] = out["company_name"].fillna("NO_COMPANY")
    return out

In [ ]:
input_path = ROOT / cfg.input_path
df = pd.read_csv(input_path)
print("Raw input:", df.shape)

df.columns = [str(c).strip() for c in df.columns]
ruleset_payload = load_ruleset_payload(cfg.ruleset_path)
df_rules, rule_waterfall = apply_ruleset_filters_light(df, ruleset_payload)
display(rule_waterfall)
print("After YAML ruleset safety filters:", df_rules.shape)

df_trf_clean = df_rules.loc[df_rules[cfg.movement_col].astype("string").eq(cfg.movement_value)].copy()
print(f"After {cfg.movement_value} filter:", df_trf_clean.shape)

for col in ["CAPTURE_DATE", "ISSUE_DATE", "TRADE_DATE", "VALUE_DATE", "MATCH_DATE"]:
    if col in df_trf_clean.columns:
        df_trf_clean[col] = pd.to_datetime(df_trf_clean[col], errors="coerce").dt.normalize()
for col in ["AMOUNT", "SIGNED_AMOUNT", "ORIGIN_AMOUNT"]:
    if col in df_trf_clean.columns:
        df_trf_clean[col] = parse_amount_series(df_trf_clean[col])

if cfg.id_col in df_trf_clean.columns:
    before = len(df_trf_clean)
    df_trf_clean = df_trf_clean.drop_duplicates(subset=cfg.id_col, keep="first").copy()
    print(f"Dropped duplicated {cfg.id_col}: {before - len(df_trf_clean)}")

company_text = first_non_empty_text(df_trf_clean, cfg.company_text_candidates)
extracted_company = company_text.map(extract_company).map(normalize_company)
if cfg.company_col in df_trf_clean.columns:
    existing_company = df_trf_clean[cfg.company_col].map(normalize_company)
else:
    existing_company = pd.Series(pd.NA, index=df_trf_clean.index, dtype="object")

df_trf_clean["company_name_raw"] = existing_company.fillna(extracted_company)
df_trf_clean["company_name"] = df_trf_clean["company_name_raw"].map(normalize_company)
df_trf_clean = canonicalize_company_names(df_trf_clean)
df_trf_clean["amount"] = pd.to_numeric(df_trf_clean[cfg.amount_col], errors="coerce")
df_trf_clean["dayofweek"] = df_trf_clean[cfg.date_col].dt.dayofweek
df_trf_clean["day_name"] = df_trf_clean[cfg.date_col].dt.day_name()
df_trf_clean["dayofmonth"] = df_trf_clean[cfg.date_col].dt.day
df_trf_clean = df_trf_clean.dropna(subset=[cfg.date_col, "amount"]).copy()

calendar = pd.date_range(df_trf_clean[cfg.date_col].min(), df_trf_clean[cfg.date_col].max(), freq="D", name=cfg.date_col)

print("Final cleaned TRF+ rows:", df_trf_clean.shape)
print("Date range:", df_trf_clean[cfg.date_col].min(), "->", df_trf_clean[cfg.date_col].max())
print("Known company rate:", round(float(df_trf_clean["company_found"].mean()), 3))

In [ ]:
def parse_payment_bin(interval) -> tuple[float, float]:
    if interval is None:
        interval = [0, np.inf]
    if len(interval) != 2:
        raise ValueError(f"payment_bin must be [min, max), got {interval!r}")
    low, high = interval
    low = -np.inf if low is None else float(low)
    high = np.inf if high is None else float(high)
    if not low < high:
        raise ValueError(f"payment_bin lower bound must be < upper bound, got {interval!r}")
    return low, high


def format_payment_bin(low: float, high: float) -> str:
    low_label = "-inf" if np.isneginf(low) else f"{low:g}"
    high_label = "inf" if np.isposinf(high) else f"{high:g}"
    return f"[{low_label}, {high_label})"


def build_manual_membership(tx: pd.DataFrame, specs: list[dict], cfg: SeriesConfig = cfg) -> pd.DataFrame:
    required = [cfg.id_col, cfg.date_col, "company_name", "amount"]
    missing = [col for col in required if col not in tx.columns]
    if missing:
        raise KeyError(f"Missing columns for manual series construction: {missing}")

    frames = []
    for spec in specs:
        series_id = str(spec["series_id"])
        company = normalize_company(spec["company"])
        if not company:
            raise ValueError(f"Could not normalize company for {series_id!r}: {spec['company']!r}")
        low, high = parse_payment_bin(spec.get("payment_bin", [0, np.inf]))
        daily_low, daily_high = parse_payment_bin(spec.get("daily_bin", [0, np.inf]))

        selected = tx.loc[
            tx["company_name"].eq(company)
            & tx["amount"].ge(low)
            & tx["amount"].lt(high)
        ].copy()
        if selected.empty:
            continue

        selected_daily_amount = selected.groupby(cfg.date_col, observed=True)["amount"].transform("sum")
        selected = selected.loc[
            selected_daily_amount.ge(daily_low)
            & selected_daily_amount.lt(daily_high)
        ].copy()

        if selected.empty:
            continue

        selected["series_id"] = series_id
        selected["company_spec"] = company

        selected["payment_bin_low"] = low
        selected["payment_bin_high"] = high
        selected["payment_bin"] = format_payment_bin(low, high)

        selected["daily_bin_low"] = daily_low
        selected["daily_bin_high"] = daily_high
        selected["daily_bin"] = format_payment_bin(daily_low, daily_high)

        selected["selected_daily_amount"] = selected.groupby(cfg.date_col, observed=True)["amount"].transform("sum")
        frames.append(
            selected[
                [
                    cfg.id_col,
                    "series_id",
                    "company_spec",
                    "company_name",
                    cfg.date_col,
                    "amount",
                    "payment_bin_low",
                    "payment_bin_high",
                    "payment_bin",
                    "daily_bin_low",
                    "daily_bin_high",
                    "daily_bin",
                    "selected_daily_amount",
                ]
            ]
        )

    columns = [
        cfg.id_col,
        "series_id",
        "company_spec",
        "company_name",
        cfg.date_col,
        "amount",
        "payment_bin_low",
        "payment_bin_high",
        "payment_bin",
        "daily_bin_low",
        "daily_bin_high",
        "daily_bin",
        "selected_daily_amount",
    ]
    if not frames:
        return pd.DataFrame(columns=columns)

    membership = pd.concat(frames, ignore_index=True)
    duplicated = membership.loc[membership.duplicated(cfg.id_col, keep=False)].sort_values(cfg.id_col)
    if not duplicated.empty:
        display(duplicated.head(50))
        raise ValueError(f"Manual series specs overlap on {duplicated[cfg.id_col].nunique()} source IDs")
    return membership


def build_dense_daily_series(
    rows: pd.DataFrame,
    calendar,
    date_col: str,
    amount_col: str = "amount",
    series_col: str = "series_id",
    series_kind: str | None = None,
) -> pd.DataFrame:
    cal = pd.DatetimeIndex(calendar, name=date_col)
    columns = [date_col, series_col, amount_col]
    if series_kind is not None:
        columns.append("series_kind")
    if rows.empty:
        return pd.DataFrame(columns=columns)

    series_ids = pd.Index(sorted(rows[series_col].dropna().astype(str).unique()), name=series_col)
    sparse = rows.groupby([date_col, series_col], observed=True)[amount_col].sum()
    full_index = pd.MultiIndex.from_product([cal, series_ids], names=[date_col, series_col])
    dense = sparse.reindex(full_index, fill_value=0.0).reset_index(name=amount_col)
    if series_kind is not None:
        dense["series_kind"] = series_kind
    return dense


def series_membership_summary(membership: pd.DataFrame, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    if membership.empty:
        return pd.DataFrame(
            columns=["series_id", "row_count", "active_days", "total_amount", "first_date", "last_date"]
        )
    return (
        membership.groupby("series_id", observed=True)
        .agg(
            row_count=(cfg.id_col, "nunique"),
            active_days=(cfg.date_col, "nunique"),
            total_amount=("amount", "sum"),
            first_date=(cfg.date_col, "min"),
            last_date=(cfg.date_col, "max"),
        )
        .reset_index()
        .sort_values("total_amount", ascending=False)
    )


def rows_for_series(series_id: str, membership: pd.DataFrame, tx: pd.DataFrame, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    ids = membership.loc[membership["series_id"].eq(series_id), cfg.id_col]
    return tx.loc[tx[cfg.id_col].isin(ids)].copy()


def exclude_series_ids(series_ids, membership: pd.DataFrame, tx: pd.DataFrame, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    ids = membership.loc[membership["series_id"].isin(series_ids), cfg.id_col]
    return tx.loc[~tx[cfg.id_col].isin(ids)].copy()

In [ ]:
manual_series_specs = [
    {
        "series_id": "EL_CORTE_INGLES",
        "company": "EL CORTE INGLES",
        "payment_bin": [0, np.inf],
        "daily_bin": [10000, np.inf],
    },
    {
        "series_id": "MERCADONA",
        "company": "MERCADONA",
        "payment_bin": [0, np.inf],
        "daily_bin": [60000, np.inf],
    },
    {
        "series_id": "IFA_RETAIL",
        "company": "IFA RETAIL",
        "payment_bin": [0, np.inf],
        "daily_bin": [0, np.inf],
    },
    {
        "series_id": "CONSUM_COOP_V",
        "company": "CONSUM COOP V",
        "payment_bin": [0, np.inf],
        "daily_bin": [80000, np.inf],
    },
    {
        "series_id": "DOUGLAS_SPAIN",
        "company": "DOUGLAS SPAIN",
        "payment_bin": [0, np.inf],
        "daily_bin": [0, np.inf],
    },
    {
        "series_id": "ADYEN",
        "company": "ADYEN",
        "payment_bin": [0, np.inf],
        "daily_bin": [0, np.inf],
    },
    {
        "series_id": "ARENAL_PERFUMERIAS",
        "company": "ARENAL PERFUMERIAS",
        "payment_bin": [0, np.inf],
        "daily_bin": [10000, np.inf],
    },
    {
        "series_id": "COFARES_SOCIEDAD_COOPERATIVA_FARMA",
        "company": "COFARES SOCIEDAD COOPERATIVA FARMA",
        "payment_bin": [0, np.inf],
        "daily_bin": [10000, np.inf],
    },
    {
        "series_id": "BANCO_DE_SABADELL",
        "company": "BANCO DE SABADELL",
        "payment_bin": [0, np.inf],
        "daily_bin": [5000, np.inf],
    },
    {
        "series_id": "REGULADORA_DE_COMPRAS_DEL",
        "company": "REGULADORA DE COMPRAS DEL",
        "payment_bin": [0, np.inf],
        # OCR note: the daily_bin line was hidden at the bottom of photo 8278.
        # Verify this value against the source notebook if it differs.
        "daily_bin": [0, np.inf],
    },
    {
        # OCR note: HOGARLIN appears in the visible plot legend, but its spec lines
        # were in the cut-off section between photos 8278 and 8279.
        "series_id": "HOGARLIN",
        "company": "HOGARLIN",
        "payment_bin": [0, np.inf],
        "daily_bin": [0, np.inf],
    },
    {
        "series_id": "SEPHORA_COSMETICOS_ESPANA_high",
        "company": "SEPHORA COSMETICOS ESPANA",
        "payment_bin": [0, np.inf],
        "daily_bin": [200000, np.inf],
    },
    {
        "series_id": "AMAZON_EU_R_L_lowp",
        "company": "AMAZON EU R L",
        "payment_bin": [0, 100000],
        "daily_bin": [0, np.inf],
    },
    {
        "series_id": "AMAZON_EU_R_L_highp",
        "company": "AMAZON EU R L",
        "payment_bin": [100000, np.inf],
        "daily_bin": [0, np.inf],
    },
    {
        "series_id": "GLOBAL_COLLECT_highp",
        "company": "GLOBAL COLLECT",
        "payment_bin": [100000, np.inf],
        "daily_bin": [0, np.inf],
    },
    {
        "series_id": "GLOBAL_COLLECT_lowp",
        "company": "GLOBAL COLLECT",
        "payment_bin": [0, 100000],
        "daily_bin": [0, np.inf],
    },
]


In [ ]:
manual_membership = build_manual_membership(df_trf_clean, manual_series_specs)
manual_ids = set(manual_membership[cfg.id_col])
clean_ids = set(df_trf_clean[cfg.id_col])
missing_manual_ids = manual_ids - clean_ids
assert not missing_manual_ids, f"Manual membership contains IDs absent from df_trf_clean: {list(missing_manual_ids)[:10]}"
assert not manual_membership[cfg.id_col].duplicated().any(), "Manual membership has duplicated source IDs"

manual_series_daily = build_dense_daily_series(
    manual_membership,
    calendar,
    date_col=cfg.date_col,
    amount_col="amount",
    series_col="series_id",
    series_kind="manual",
)

df_trf_after_manual = df_trf_clean.loc[~df_trf_clean[cfg.id_col].isin(manual_ids)].copy()
assert len(df_trf_clean) - len(df_trf_after_manual) == len(manual_ids)

print("Manual selected IDs:", len(manual_ids))
print("Rows left for automatic clustering:", len(df_trf_after_manual))

display(manual_membership.head(5))
manual_membership_summary = series_membership_summary(manual_membership)
display(manual_membership_summary)
display(manual_series_daily.head(2))

In [ ]:
#series_name = "EL_CORTE_INGLES_FULL"
#series = manual_series_daily[manual_series_daily["series_id"].eq(series_name)]

fig = px.line(
    # series,
    manual_series_daily,
    x=cfg.date_col,
    y="amount",
    color="series_id",
    markers=False,
    labels={cfg.date_col: "Value date", "amount": "Daily amount"},
)
fig.update_layout(template="plotly_white", height=430, legend_title_text="")
fig.show()

# Other companies

The cells below keep amount-band setup shared, then split company-regime and transaction-regime exploration into independent sections.


In [ ]:

MIN_ACTIVE_DAYS_FOR_CLUSTER = 3
AUTO_AMOUNT_BANDS = [
    (0, 20_000),
    (20_000, 100_000),
    (100_000, 500_000),
    (500_000, 1_000_000),
    (1_000_000, np.inf),
]
AUTO_AMOUNT_BAND_LABELS = ("tiny", "low", "mid", "large", "mega")
AUTO_BAND_CLUSTER_COUNTS = {"tiny": 2, "low": 3, "mid": 4, "large": 3, "mega": 2}
PLOT_SERIES_IDS = []


def normalize_auto_amount_bands(amount_bands, labels=None) -> tuple[list[str], list[tuple[float, float]]]:
    bands = [(float(low), float(high)) for low, high in amount_bands]
    if not bands:
        raise ValueError("AUTO_AMOUNT_BANDS must contain at least one interval")
    for i, (low, high) in enumerate(bands):
        if not low < high:
            raise ValueError(f"Amount band {i} lower bound must be < upper bound: {(low, high)!r}")
        if i and not np.isclose(low, bands[i - 1][1]):
            raise ValueError("AUTO_AMOUNT_BANDS must be contiguous so every row can be routed exactly once")
    if labels is None:
        labels = [f"band_{i + 1:02d}" for i in range(len(bands))]
    labels = [str(label) for label in labels]
    if len(labels) != len(bands):
        raise ValueError("AUTO_AMOUNT_BAND_LABELS must have the same length as AUTO_AMOUNT_BANDS")
    if len(set(labels)) != len(labels):
        raise ValueError("AUTO_AMOUNT_BAND_LABELS must be unique")
    return labels, bands


def format_auto_amount_band(low: float, high: float) -> str:
    low_label = f"{low:,.0f}" if np.isfinite(low) else "-inf"
    high_label = f"{high:,.0f}" if np.isfinite(high) else "inf"
    return f"[{low_label}, {high_label})"


def add_amount_band_columns(
    tx: pd.DataFrame,
    amount_bands=AUTO_AMOUNT_BANDS,
    labels=AUTO_AMOUNT_BAND_LABELS,
) -> pd.DataFrame:
    out = tx.copy()
    band_labels, bands = normalize_auto_amount_bands(amount_bands, labels)
    band_order = {label: i for i, label in enumerate(band_labels)}
    band_display = {label: format_auto_amount_band(low, high) for label, (low, high) in zip(band_labels, bands)}

    out["amount_abs"] = pd.to_numeric(out["amount"], errors="coerce").abs()
    out["amount_band"] = pd.NA
    for label, (low, high) in zip(band_labels, bands):
        mask = out["amount_abs"].ge(low) & out["amount_abs"].lt(high)
        out.loc[mask, "amount_band"] = label

    if out["amount_band"].isna().any():
        bad = out.loc[out["amount_band"].isna(), [cfg.id_col, "amount", "amount_abs"]].head(10)
        display(bad)
        raise ValueError("Some rows could not be assigned to an automatic amount band")

    out["amount_band"] = out["amount_band"].astype(str)
    out["amount_band_order"] = out["amount_band"].map(band_order).astype(int)
    out["amount_band_display"] = out["amount_band"].map(band_display)
    return out


def current_amount_band_table(amount_bands=AUTO_AMOUNT_BANDS, labels=AUTO_AMOUNT_BAND_LABELS) -> pd.DataFrame:
    band_labels, bands = normalize_auto_amount_bands(amount_bands, labels)
    return pd.DataFrame(
        {
            "amount_band": band_labels,
            "amount_band_order": range(len(band_labels)),
            "amount_band_display": [format_auto_amount_band(low, high) for low, high in bands],
            "low": [low for low, _ in bands],
            "high": [high for _, high in bands],
        }
    )


def interval_candidate_table(method: str, edges, labels=AUTO_AMOUNT_BAND_LABELS) -> pd.DataFrame:
    clean_edges = [float(edge) for edge in edges]
    rows = []
    for i, label in enumerate(labels):
        low = clean_edges[i]
        high = clean_edges[i + 1]
        rows.append(
            {
                "method": method,
                "amount_band": label,
                "amount_band_order": i,
                "low": low,
                "high": high,
                "amount_band_display": format_auto_amount_band(low, high),
            }
        )
    return pd.DataFrame(rows)


def weighted_quantile(values, weights, quantiles) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights >= 0)
    values = values[valid]
    weights = weights[valid]
    if len(values) == 0 or weights.sum() <= 0:
        return np.full(len(quantiles), np.nan)
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cumulative = np.cumsum(weights)
    cumulative = cumulative / cumulative[-1]
    return np.interp(quantiles, cumulative, values)


def amount_band_threshold_diagnostics(tx: pd.DataFrame, labels=AUTO_AMOUNT_BAND_LABELS) -> tuple[pd.DataFrame, pd.DataFrame]:
    values = pd.to_numeric(tx["amount_abs"], errors="coerce").dropna().astype(float)
    quantile_points = [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
    quantiles = pd.DataFrame(
        {
            "quantile": quantile_points,
            "amount_abs": [float(values.quantile(q)) if len(values) else np.nan for q in quantile_points],
        }
    )
    if values.empty:
        return quantiles, pd.DataFrame()

    n_bands = len(labels)
    candidate_frames = [
        interval_candidate_table("current_hardcoded", [low for low, _ in AUTO_AMOUNT_BANDS] + [AUTO_AMOUNT_BANDS[-1][1]], labels)
    ]

    equal_edges = values.quantile(np.linspace(0, 1, n_bands + 1)).to_numpy(dtype=float).copy()
    equal_edges[0] = 0.0
    equal_edges[-1] = np.inf
    candidate_frames.append(interval_candidate_table("equal_frequency", equal_edges, labels))

    positive = values.loc[values.gt(0)]
    if not positive.empty:
        start = max(float(positive.quantile(0.01)), 1.0)
        stop = max(float(positive.quantile(0.99)), start)
        internal = np.geomspace(start, stop, max(n_bands - 1, 1))
        log_edges = [0.0, *internal.tolist(), np.inf]
        candidate_frames.append(interval_candidate_table("log_spaced_p01_p99", log_edges, labels))

    weighted_internal = weighted_quantile(values, values.clip(lower=0), np.linspace(0, 1, n_bands + 1)[1:-1])
    if np.isfinite(weighted_internal).all():
        weighted_edges = [0.0, *weighted_internal.tolist(), np.inf]
        candidate_frames.append(interval_candidate_table("weighted_by_abs_amount", weighted_edges, labels))

    return quantiles, pd.concat(candidate_frames, ignore_index=True, sort=False)


def summarize_amount_bands(tx: pd.DataFrame, calendar) -> pd.DataFrame:
    columns = [
        "amount_band",
        "amount_band_order",
        "amount_band_display",
        "row_count",
        "company_count",
        "active_days",
        "zero_rate",
        "total_amount",
        "total_abs_amount",
        "q50_row_abs_amount",
        "q90_row_abs_amount",
        "max_row_abs_amount",
    ]
    if tx.empty:
        return pd.DataFrame(columns=columns)
    work = tx.copy()
    summary = (
        work.groupby(["amount_band", "amount_band_order", "amount_band_display"], observed=True)
        .agg(
            row_count=(cfg.id_col, "nunique"),
            company_count=("company_key", "nunique"),
            active_days=(cfg.date_col, "nunique"),
            total_amount=("amount", "sum"),
            total_abs_amount=("amount_abs", "sum"),
            q50_row_abs_amount=("amount_abs", lambda s: float(s.quantile(0.50))),
            q90_row_abs_amount=("amount_abs", lambda s: float(s.quantile(0.90))),
            max_row_abs_amount=("amount_abs", "max"),
        )
        .reset_index()
        .sort_values("amount_band_order")
    )
    summary["zero_rate"] = 1.0 - summary["active_days"] / max(len(calendar), 1)
    return summary[columns]


def prepare_cluster_matrix(
    features: pd.DataFrame,
    categorical_columns=("sbc_label", "amount_band", "company_sbc_label"),
    exclude_columns=None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    exclude_columns = set(exclude_columns or [])
    numeric = features.select_dtypes(include=[np.number]).copy()
    numeric = numeric.drop(columns=[col for col in exclude_columns if col in numeric.columns], errors="ignore")
    numeric = numeric.replace([np.inf, -np.inf], np.nan)
    log_tokens = ("count", "days", "amount", "gap", "run", "diff", "row")
    for col in numeric.columns:
        values = numeric[col].dropna()
        if not values.empty and values.min() >= 0 and any(token in col for token in log_tokens):
            numeric[col] = np.log1p(numeric[col])
    medians = numeric.median(numeric_only=True).fillna(0.0)
    numeric = numeric.fillna(medians).fillna(0.0)
    categorical_frames = []
    for col in categorical_columns:
        if col in features.columns:
            categorical_frames.append(pd.get_dummies(features[col].fillna("unknown").astype(str), prefix=col, dtype=float))
    matrix = pd.concat([numeric, *categorical_frames], axis=1)
    if matrix.empty:
        matrix["constant"] = 1.0
    scaler = RobustScaler()
    scaled = pd.DataFrame(scaler.fit_transform(matrix), columns=matrix.columns, index=features.index)
    return matrix, scaled


def add_pca_coordinates(clustered: pd.DataFrame, scaled: pd.DataFrame) -> pd.DataFrame:
    out = clustered.copy()
    coords = np.zeros((len(out), 2), dtype=float)
    if len(out) >= 2 and scaled.shape[1] >= 2:
        coords = PCA(n_components=2).fit_transform(scaled)
    elif len(out) and scaled.shape[1] == 1:
        coords[:, 0] = scaled.iloc[:, 0].to_numpy()
    out["pca_x"] = coords[:, 0]
    out["pca_y"] = coords[:, 1]
    size_source = out["total_abs_amount"] if "total_abs_amount" in out.columns else out.get("amount_abs", pd.Series(1.0, index=out.index))
    out["plot_size"] = size_source.clip(lower=0)
    return out


def validate_strategy_membership(membership: pd.DataFrame, tx: pd.DataFrame, label: str, cfg: SeriesConfig = cfg) -> None:
    source_ids = set(tx[cfg.id_col])
    membership_ids = set(membership[cfg.id_col])
    missing = source_ids - membership_ids
    extra = membership_ids - source_ids
    if missing or extra:
        raise AssertionError(f"{label} membership mismatch: missing={len(missing)}, extra={len(extra)}")
    duplicated = membership.loc[membership.duplicated(cfg.id_col, keep=False)].sort_values(cfg.id_col)
    if not duplicated.empty:
        display(duplicated.head(50))
        raise AssertionError(f"{label} assigns some source IDs more than once")


def validate_dense_daily(dense: pd.DataFrame, calendar, label: str, series_col: str = "series_id") -> None:
    if dense.empty:
        return
    expected_rows = len(calendar) * dense[series_col].nunique()
    assert len(dense) == expected_rows, f"{label} should have one row per day per series"


def build_band_total_daily_series(
    membership: pd.DataFrame,
    calendar,
    prefix: str,
    series_kind: str,
    date_col: str = cfg.date_col,
) -> pd.DataFrame:
    columns = [date_col, "series_id", "amount", "amount_band", "amount_band_order", "series_kind"]
    if membership.empty:
        return pd.DataFrame(columns=columns)
    cal = pd.DatetimeIndex(calendar, name=date_col)
    band_meta = (
        membership[["amount_band", "amount_band_order"]]
        .drop_duplicates()
        .sort_values("amount_band_order")
        .reset_index(drop=True)
    )
    sparse = membership.groupby([date_col, "amount_band"], observed=True)["amount"].sum()
    full_index = pd.MultiIndex.from_product([cal, band_meta["amount_band"]], names=[date_col, "amount_band"])
    dense = sparse.reindex(full_index, fill_value=0.0).reset_index(name="amount")
    dense = dense.merge(band_meta, on="amount_band", how="left", validate="many_to_one")
    dense["series_id"] = prefix + "_" + dense["amount_band"].str.upper() + "_TOTAL"
    dense["series_kind"] = series_kind
    return dense[columns]


def build_series_summary(
    membership: pd.DataFrame,
    dense_daily: pd.DataFrame,
    strategy_label: str,
    cfg: SeriesConfig = cfg,
) -> pd.DataFrame:
    columns = [
        "strategy",
        "series_id",
        "amount_band",
        "amount_band_order",
        "company_count",
        "row_count",
        "active_days",
        "zero_rate",
        "total_amount",
        "total_abs_amount",
        "max_daily_abs_amount",
        "q90_abs_daily_amount",
        "max_day_abs_share",
        "top_company_share",
        "first_date",
        "last_date",
    ]
    if membership.empty:
        return pd.DataFrame(columns=columns)
    work = membership.copy()
    work["abs_amount"] = work["amount"].abs()
    base = (
        work.groupby("series_id", observed=True)
        .agg(
            company_count=("company_key", "nunique"),
            row_count=(cfg.id_col, "nunique"),
            total_amount=("amount", "sum"),
            total_abs_amount=("abs_amount", "sum"),
            first_date=(cfg.date_col, "min"),
            last_date=(cfg.date_col, "max"),
        )
        .reset_index()
    )
    band_meta = (
        work.sort_values(["series_id", "amount_band_order"])
        .groupby("series_id", observed=True)
        .agg(amount_band=("amount_band", "first"), amount_band_order=("amount_band_order", "first"))
        .reset_index()
    )
    dense = dense_daily.copy()
    dense["daily_abs_amount"] = dense["amount"].abs()
    dense_stats = (
        dense.groupby("series_id", observed=True)
        .agg(
            calendar_days=(cfg.date_col, "count"),
            active_days=("daily_abs_amount", lambda s: int(s.ne(0).sum())),
            zero_days=("daily_abs_amount", lambda s: int(s.eq(0).sum())),
            max_daily_abs_amount=("daily_abs_amount", "max"),
            q90_abs_daily_amount=("daily_abs_amount", lambda s: float(s.quantile(0.90))),
            total_abs_daily_amount=("daily_abs_amount", "sum"),
        )
        .reset_index()
    )
    dense_stats["zero_rate"] = dense_stats["zero_days"] / dense_stats["calendar_days"].replace(0, np.nan)
    dense_stats["max_day_abs_share"] = dense_stats["max_daily_abs_amount"] / dense_stats["total_abs_daily_amount"].replace(0, np.nan)
    company_abs = work.groupby(["series_id", "company_key"], observed=True)["abs_amount"].sum()
    top_company_abs = company_abs.groupby(level=0).max().rename("top_company_abs").reset_index()
    summary = base.merge(band_meta, on="series_id", how="left")
    summary = summary.merge(dense_stats, on="series_id", how="left")
    summary = summary.merge(top_company_abs, on="series_id", how="left")
    summary["top_company_share"] = summary["top_company_abs"] / summary["total_abs_amount"].replace(0, np.nan)
    summary["strategy"] = strategy_label
    return summary[columns].sort_values(["amount_band_order", "series_id"]).reset_index(drop=True)


def membership_view_for_band_totals(membership: pd.DataFrame, prefix: str) -> pd.DataFrame:
    if membership.empty:
        return membership.copy()
    view = membership.copy()
    view["series_id"] = prefix + "_" + view["amount_band"].str.upper() + "_TOTAL"
    return view


def build_series_company_detail(membership: pd.DataFrame, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    columns = ["series_id", "amount_band", "amount_band_order", "company_key", "row_count", "active_days", "total_amount", "total_abs_amount", "first_date", "last_date"]
    if membership.empty:
        return pd.DataFrame(columns=columns)
    work = membership.copy()
    work["abs_amount"] = work["amount"].abs()
    return (
        work.groupby(["series_id", "amount_band", "amount_band_order", "company_key"], observed=True)
        .agg(
            row_count=(cfg.id_col, "nunique"),
            active_days=(cfg.date_col, "nunique"),
            total_amount=("amount", "sum"),
            total_abs_amount=("abs_amount", "sum"),
            first_date=(cfg.date_col, "min"),
            last_date=(cfg.date_col, "max"),
        )
        .reset_index()
        .sort_values(["series_id", "total_abs_amount"], ascending=[True, False])
        .reset_index(drop=True)
    )


def plot_regime_daily(series_daily: pd.DataFrame, summary: pd.DataFrame, title: str) -> None:
    if series_daily.empty:
        print("No generated series to plot.")
        return
    plot_daily = series_daily.merge(
        summary[["series_id", "amount_band", "zero_rate", "top_company_share", "max_day_abs_share"]],
        on="series_id",
        how="left",
        validate="many_to_one",
    )
    fig = px.line(
        plot_daily,
        x=cfg.date_col,
        y="amount",
        color="series_id",
        facet_row="amount_band",
        markers=False,
        labels={cfg.date_col: "Value date", "amount": "Daily amount", "amount_band": "Amount band"},
        title=title,
        hover_data=["zero_rate", "top_company_share", "max_day_abs_share"],
        category_orders={"amount_band": list(AUTO_AMOUNT_BAND_LABELS)},
    )
    fig.update_yaxes(matches=None)
    fig.update_layout(template="plotly_white", height=max(520, 190 * plot_daily["amount_band"].nunique()), legend_title_text="")
    fig.show()


def plot_band_totals(band_daily: pd.DataFrame, title: str) -> None:
    if band_daily.empty:
        print("No band-total series to plot.")
        return
    fig = px.line(
        band_daily,
        x=cfg.date_col,
        y="amount",
        color="series_id",
        markers=False,
        labels={cfg.date_col: "Value date", "amount": "Daily amount"},
        title=title,
        category_orders={"amount_band": list(AUTO_AMOUNT_BAND_LABELS)},
    )
    fig.update_layout(template="plotly_white", height=430, legend_title_text="")
    fig.show()


def plot_cluster_map(clusters: pd.DataFrame, title: str) -> None:
    if clusters.empty:
        print("No clusters to plot.")
        return
    scatter_frame = clusters.copy()
    if len(scatter_frame) > 5000:
        scatter_frame = scatter_frame.sample(5000, random_state=0)
    hover_candidates = [
        "company_key",
        cfg.id_col,
        "row_count",
        "active_days",
        "amount_abs",
        "total_abs_amount",
        "active_rate",
        "adi",
        "cv2",
        "sbc_label",
        "company_sbc_label",
        "cluster_kind",
    ]
    hover_cols = [col for col in hover_candidates if col in scatter_frame.columns]
    fig = px.scatter(
        scatter_frame,
        x="pca_x",
        y="pca_y",
        color="auto_series_id",
        symbol="amount_band" if "amount_band" in scatter_frame.columns else None,
        size="plot_size" if "plot_size" in scatter_frame.columns else None,
        size_max=24,
        hover_name="company_key" if "company_key" in scatter_frame.columns else None,
        hover_data=hover_cols,
        category_orders={"amount_band": list(AUTO_AMOUNT_BAND_LABELS)},
        labels={"pca_x": "PCA 1", "pca_y": "PCA 2", "auto_series_id": "Auto series"},
        title=title,
    )
    fig.update_layout(template="plotly_white", height=560, legend_title_text="")
    fig.show()


def plot_volume_bar(summary: pd.DataFrame, title: str) -> None:
    if summary.empty:
        print("No summary to plot.")
        return
    fig = px.bar(
        summary,
        x="series_id",
        y="total_abs_amount",
        color="amount_band",
        text="company_count",
        hover_data=["row_count", "active_days", "zero_rate", "max_day_abs_share", "top_company_share"],
        labels={"series_id": "Series", "total_abs_amount": "Total absolute amount", "company_count": "Companies"},
        title=title,
        category_orders={"amount_band": list(AUTO_AMOUNT_BAND_LABELS)},
    )
    fig.update_layout(template="plotly_white", height=460, xaxis_tickangle=-35)
    fig.show()


def plot_cluster_heatmap(profile: pd.DataFrame, title: str) -> None:
    if profile.empty:
        print("No cluster profile to plot.")
        return
    profile_plot = profile.set_index("auto_series_id")
    plot_cols = profile_plot.var(axis=0).sort_values(ascending=False).head(30).index.tolist()
    if not plot_cols:
        print("No varying profile columns to plot.")
        return
    fig = px.imshow(
        profile_plot[plot_cols],
        aspect="auto",
        color_continuous_scale="RdBu_r",
        zmin=-3,
        zmax=3,
        labels={"x": "Scaled feature", "y": "Auto series", "color": "Median scaled value"},
        title=title,
    )
    fig.update_layout(template="plotly_white", height=max(360, 42 * len(profile_plot)))
    fig.show()


df_trf_after_manual_banded = add_amount_band_columns(df_trf_after_manual)
auto_current_band_summary = summarize_amount_bands(df_trf_after_manual_banded, calendar)
auto_amount_quantiles, auto_band_threshold_candidates = amount_band_threshold_diagnostics(df_trf_after_manual_banded)

display(current_amount_band_table())
display(auto_current_band_summary)
display(auto_amount_quantiles)
display(auto_band_threshold_candidates)

if not df_trf_after_manual_banded.empty:
    fig = px.bar(
        auto_current_band_summary,
        x="amount_band",
        y="total_abs_amount",
        color="amount_band",
        text="row_count",
        hover_data=["company_count", "active_days", "q90_row_abs_amount", "max_row_abs_amount"],
        title="Current hardcoded amount bands",
        category_orders={"amount_band": list(AUTO_AMOUNT_BAND_LABELS)},
    )
    fig.update_layout(template="plotly_white", height=420, showlegend=False)
    fig.show()

    positive_amounts = df_trf_after_manual_banded.loc[df_trf_after_manual_banded["amount_abs"].gt(0)].copy()
    if not positive_amounts.empty:
        fig = px.histogram(
            positive_amounts,
            x="amount_abs",
            color="amount_band",
            nbins=80,
            log_x=True,
            labels={"amount_abs": "Absolute row amount"},
            title="Residual transaction amount distribution by current band",
            category_orders={"amount_band": list(AUTO_AMOUNT_BAND_LABELS)},
        )
        fig.update_layout(template="plotly_white", height=430)
        fig.show()


## Company Regime Approach

Logic: one clustering unit is a `company_key + amount_band` regime. All rows from the same company in the same amount band move together into one generated series. This keeps company behavior coherent while still allowing a company to have separate low, mid, and high amount regimes.


In [ ]:

def build_company_regime_feature_frame(tx: pd.DataFrame, calendar, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    cal = pd.DatetimeIndex(calendar, name=cfg.date_col)
    n_days = len(cal)
    rows = []
    if tx.empty:
        return pd.DataFrame()

    grouped = tx.groupby(["company_key", "amount_band", "amount_band_order"], dropna=False, observed=True)
    for (company_key, amount_band, amount_band_order), group in grouped:
        company_key = "NO_COMPANY" if pd.isna(company_key) else str(company_key)
        amount_band = str(amount_band)
        daily = group.groupby(cfg.date_col, observed=True)["amount"].sum().reindex(cal, fill_value=0.0).astype(float)
        row_counts_by_day = group.groupby(cfg.date_col, observed=True)[cfg.id_col].nunique().reindex(cal, fill_value=0)
        active_mask = daily.ne(0.0)
        active_days = int(active_mask.sum())
        active_amounts = daily.loc[active_mask]
        active_abs_amounts = active_amounts.abs()
        total_amount = float(daily.sum())
        total_abs_amount = float(daily.abs().sum())
        abs_daily = daily.abs().sort_values(ascending=False)
        gaps = gap_stats(daily.index[active_mask])
        adi = float(n_days / active_days) if active_days else np.nan
        cv2 = safe_cv2(active_abs_amounts) if active_days else np.nan

        if active_days:
            active_dates = pd.DatetimeIndex(active_amounts.index)
            weekday_values = active_dates.dayofweek
            dayofmonth_values = active_dates.day
            rows_per_active_day = row_counts_by_day.loc[active_mask].astype(float)
            active_q50 = float(active_abs_amounts.quantile(0.50))
            active_q90 = float(active_abs_amounts.quantile(0.90))
            active_q95 = float(active_abs_amounts.quantile(0.95))
            active_max = float(active_abs_amounts.max())
            month_start_share = float((active_dates.day <= 5).mean())
            month_end_share = float((active_dates.day >= (active_dates.days_in_month - 4)).mean())
            weekend_share = float((active_dates.dayofweek >= 5).mean())
            spike_active_share = float(active_abs_amounts.ge(active_q95).mean())
        else:
            weekday_values = []
            dayofmonth_values = []
            rows_per_active_day = pd.Series(dtype=float)
            active_q50 = active_q90 = active_q95 = active_max = 0.0
            month_start_share = month_end_share = weekend_share = spike_active_share = 0.0

        lead_days = pd.Series(dtype=float)
        if "TRADE_DATE" in group.columns:
            lead_days = (group[cfg.date_col] - group["TRADE_DATE"]).dt.days.dropna().astype(float)

        row = {
            "unit_key": f"{company_key}|{amount_band}",
            "company_key": company_key,
            "amount_band": amount_band,
            "amount_band_order": int(amount_band_order),
            "company_found": bool(group["company_found"].fillna(False).any()) if "company_found" in group.columns else company_key != "NO_COMPANY",
            "row_count": int(group[cfg.id_col].nunique()),
            "active_days": active_days,
            "calendar_days": n_days,
            "total_amount": total_amount,
            "total_abs_amount": total_abs_amount,
            "mean_calendar_daily_amount": float(daily.mean()),
            "mean_active_abs_daily_amount": float(active_abs_amounts.mean()) if active_days else 0.0,
            "median_active_abs_daily_amount": active_q50,
            "q90_active_abs_daily_amount": active_q90,
            "q95_active_abs_daily_amount": active_q95,
            "max_active_abs_daily_amount": active_max,
            "max_day_abs_share": float(abs_daily.iloc[0] / total_abs_amount) if total_abs_amount else 0.0,
            "top3_day_abs_share": float(abs_daily.head(3).sum() / total_abs_amount) if total_abs_amount else 0.0,
            "active_rate": float(active_days / n_days) if n_days else 0.0,
            "zero_rate": float(1.0 - active_days / n_days) if n_days else 0.0,
            "adi": adi,
            "cv2": cv2,
            "longest_zero_run": longest_run(~active_mask.to_numpy()),
            "longest_active_run": longest_run(active_mask.to_numpy()),
            "weekday_entropy": normalized_entropy(weekday_values),
            "weekday_top_share": top_share(weekday_values),
            "weekend_active_share": weekend_share,
            "month_start_active_share": month_start_share,
            "month_end_active_share": month_end_share,
            "dayofmonth_entropy": normalized_entropy(dayofmonth_values),
            "dayofmonth_top_share": top_share(dayofmonth_values),
            "rows_per_active_day_mean": float(rows_per_active_day.mean()) if active_days else 0.0,
            "rows_per_active_day_max": float(rows_per_active_day.max()) if active_days else 0.0,
            "rows_per_active_day_cv2": safe_cv2(rows_per_active_day),
            "autocorr_lag_1": autocorr_at_lag(daily, 1),
            "autocorr_lag_7": autocorr_at_lag(daily, 7),
            "autocorr_lag_14": autocorr_at_lag(daily, 14),
            "autocorr_lag_28": autocorr_at_lag(daily, 28),
            "mean_abs_daily_diff": float(daily.diff().abs().dropna().mean()) if n_days > 1 else 0.0,
            "spike_active_share": spike_active_share,
            "recent_30_amount_share": share_last_days(daily.abs(), 30),
            "recent_90_amount_share": share_last_days(daily.abs(), 90),
            "recent_180_amount_share": share_last_days(daily.abs(), 180),
            "lead_days_mean": float(lead_days.mean()) if not lead_days.empty else np.nan,
            "lead_days_median": float(lead_days.median()) if not lead_days.empty else np.nan,
            "known_at_least_d1_share": float(lead_days.ge(1).mean()) if not lead_days.empty else np.nan,
            "known_same_day_share": float(lead_days.eq(0).mean()) if not lead_days.empty else np.nan,
        }
        row.update(gaps)
        row["sbc_label"] = sbc_label(row["adi"], row["cv2"])
        rows.append(row)

    return pd.DataFrame(rows).sort_values(["amount_band_order", "total_abs_amount"], ascending=[True, False]).reset_index(drop=True)


def cluster_company_regime_features(
    features: pd.DataFrame,
    cluster_counts=AUTO_BAND_CLUSTER_COUNTS,
    min_active_days: int = MIN_ACTIVE_DAYS_FOR_CLUSTER,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if features.empty:
        return features.copy(), pd.DataFrame(), pd.DataFrame()

    _, scaled = prepare_cluster_matrix(features)
    clustered = features.copy()
    clustered["auto_series_id"] = pd.NA
    clustered["cluster_raw"] = pd.NA
    clustered["cluster_kind"] = "clustered"
    band_values = sorted(
        clustered["amount_band"].dropna().unique(),
        key=lambda value: int(clustered.loc[clustered["amount_band"].eq(value), "amount_band_order"].iloc[0]),
    )

    for band in band_values:
        band_mask = clustered["amount_band"].eq(band)
        sparse_mask = band_mask & clustered["active_days"].lt(min_active_days)
        if sparse_mask.any():
            sparse_label = f"COMPANY_SP_{str(band).upper()}"
            clustered.loc[sparse_mask, "auto_series_id"] = sparse_label
            clustered.loc[sparse_mask, "cluster_raw"] = f"{band}_sparse"
            clustered.loc[sparse_mask, "cluster_kind"] = "sparse"

        eligible_idx = clustered.index[band_mask & ~sparse_mask].tolist()
        if not eligible_idx:
            continue
        k = min(max(1, int(cluster_counts.get(str(band), 1))), len(eligible_idx))
        if k == 1:
            raw_labels = np.zeros(len(eligible_idx), dtype=int)
        else:
            raw_labels = KMeans(n_clusters=k, random_state=0, n_init=50).fit_predict(scaled.loc[eligible_idx])
        raw_series = pd.Series(raw_labels, index=eligible_idx, name="cluster_raw")
        order_frame = clustered.loc[eligible_idx, ["total_abs_amount"]].assign(cluster_raw=raw_series)
        cluster_order = order_frame.groupby("cluster_raw", observed=True)["total_abs_amount"].sum().sort_values(ascending=False).index.tolist()
        label_map = {raw: f"COMPANY_{str(band).upper()}_{i + 1:02d}" for i, raw in enumerate(cluster_order)}
        clustered.loc[eligible_idx, "cluster_raw"] = [f"{band}_{raw}" for raw in raw_labels]
        clustered.loc[eligible_idx, "auto_series_id"] = [label_map[raw] for raw in raw_labels]

    if clustered["auto_series_id"].isna().any():
        raise ValueError("Some company-regime units were not assigned")

    clustered = add_pca_coordinates(clustered, scaled)
    profile = scaled.assign(auto_series_id=clustered["auto_series_id"].astype(str).to_numpy()).groupby("auto_series_id", observed=True).median().reset_index()
    return clustered, scaled, profile


company_regime_features = build_company_regime_feature_frame(df_trf_after_manual_banded, calendar)
company_regime_clusters, company_regime_feature_matrix_scaled, company_regime_cluster_profile_scaled = cluster_company_regime_features(company_regime_features)

if company_regime_clusters.empty:
    company_regime_membership = pd.DataFrame(columns=[cfg.id_col, "series_id", "company_key", "amount_band", "amount_band_order", cfg.date_col, "amount", "amount_abs", "cluster_kind", "unit_key"])
else:
    company_regime_membership = df_trf_after_manual_banded.merge(
        company_regime_clusters[["company_key", "amount_band", "unit_key", "auto_series_id", "cluster_kind"]],
        on=["company_key", "amount_band"],
        how="inner",
        validate="many_to_one",
    ).rename(columns={"auto_series_id": "series_id"})
    company_regime_membership = company_regime_membership[
        [cfg.id_col, "series_id", "company_key", "amount_band", "amount_band_order", cfg.date_col, "amount", "amount_abs", "cluster_kind", "unit_key"]
    ]

validate_strategy_membership(company_regime_membership, df_trf_after_manual_banded, "company_regime")
company_regime_series_daily = build_dense_daily_series(
    company_regime_membership,
    calendar,
    date_col=cfg.date_col,
    amount_col="amount",
    series_col="series_id",
    series_kind="company_regime",
)
company_regime_band_daily = build_band_total_daily_series(
    company_regime_membership,
    calendar,
    prefix="COMPANY_BAND",
    series_kind="company_regime_band_total",
)
company_regime_all_daily = pd.concat([company_regime_series_daily, company_regime_band_daily], ignore_index=True, sort=False)

validate_dense_daily(company_regime_series_daily, calendar, "company_regime_series_daily")
validate_dense_daily(company_regime_band_daily, calendar, "company_regime_band_daily")

company_regime_series_summary = build_series_summary(company_regime_membership, company_regime_series_daily, "company_regime")
company_regime_band_summary = build_series_summary(
    membership_view_for_band_totals(company_regime_membership, "COMPANY_BAND"),
    company_regime_band_daily,
    "company_regime_band_total",
)
company_regime_series_company_detail = build_series_company_detail(company_regime_membership)
company_regime_company_series_detail = company_regime_series_company_detail.sort_values(
    ["company_key", "amount_band_order", "series_id"], ascending=[True, True, True]
).reset_index(drop=True)
company_regime_band_company_detail = build_series_company_detail(membership_view_for_band_totals(company_regime_membership, "COMPANY_BAND"))

print("Company regime rows:", len(company_regime_membership))
print("Company regime companies:", company_regime_membership["company_key"].nunique() if not company_regime_membership.empty else 0)
print("Company regime generated series:", company_regime_series_summary["series_id"].nunique() if not company_regime_series_summary.empty else 0)
print("Company regime band-total series:", company_regime_band_daily["series_id"].nunique() if not company_regime_band_daily.empty else 0)


In [ ]:

display(company_regime_series_summary)
display(company_regime_band_summary)
display(company_regime_series_company_detail)
display(company_regime_company_series_detail)
display(company_regime_band_company_detail)
display(company_regime_clusters.head(30))
display(company_regime_all_daily.head())


In [ ]:

plot_regime_daily(company_regime_series_daily, company_regime_series_summary, "Company regime generated daily series by amount band")
plot_band_totals(company_regime_band_daily, "Company regime band-total daily series")
plot_cluster_map(company_regime_clusters, "Company regime clustering map")
plot_volume_bar(company_regime_series_summary, "Company regime series volume and concentration")
plot_cluster_heatmap(company_regime_cluster_profile_scaled, "Company regime cluster profile heatmap")

if PLOT_SERIES_IDS:
    plot_series = company_regime_all_daily.loc[company_regime_all_daily["series_id"].isin(PLOT_SERIES_IDS)].copy()
    if not plot_series.empty:
        fig = px.line(
            plot_series,
            x=cfg.date_col,
            y="amount",
            color="series_id",
            labels={cfg.date_col: "Value date", "amount": "Daily amount", "series_id": "Series"},
            title="Selected company-regime series",
        )
        fig.update_layout(template="plotly_white", height=430, legend_title_text="")
        fig.show()


## Transaction Regime Approach

Logic: one clustering unit is a transaction row. Rows are first routed into hard amount bands, then clustered inside each band using row amount, calendar position, lead-time signals, same-day company totals, and company-history features. This is more flexible than company regime: the same company can be split across multiple generated series, even within the same amount band.


In [ ]:

def build_company_history_features(tx: pd.DataFrame, calendar, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    cal = pd.DatetimeIndex(calendar, name=cfg.date_col)
    n_days = len(cal)
    rows = []
    if tx.empty:
        return pd.DataFrame(columns=["company_key"])

    for company_key, group in tx.groupby("company_key", dropna=False, observed=True):
        company_key = "NO_COMPANY" if pd.isna(company_key) else str(company_key)
        daily = group.groupby(cfg.date_col, observed=True)["amount"].sum().reindex(cal, fill_value=0.0).astype(float)
        active_mask = daily.ne(0.0)
        active_days = int(active_mask.sum())
        active_abs = daily.loc[active_mask].abs()
        total_abs = float(daily.abs().sum())
        cv2 = safe_cv2(active_abs) if active_days else np.nan
        adi = float(n_days / active_days) if active_days else np.nan
        rows.append(
            {
                "company_key": company_key,
                "company_row_count": int(group[cfg.id_col].nunique()),
                "company_active_days": active_days,
                "company_total_abs_amount": total_abs,
                "company_active_rate": float(active_days / n_days) if n_days else 0.0,
                "company_adi": adi,
                "company_cv2": cv2,
                "company_sbc_label": sbc_label(adi, cv2),
                "company_q90_abs_daily_amount": float(active_abs.quantile(0.90)) if active_days else 0.0,
                "company_max_abs_daily_amount": float(active_abs.max()) if active_days else 0.0,
                "company_max_day_abs_share": float(daily.abs().max() / total_abs) if total_abs else 0.0,
            }
        )
    return pd.DataFrame(rows)


def build_transaction_feature_frame(tx: pd.DataFrame, calendar, cfg: SeriesConfig = cfg) -> pd.DataFrame:
    work = tx.copy()
    if work.empty:
        return pd.DataFrame()

    company_history = build_company_history_features(work, calendar, cfg)
    company_day = (
        work.groupby(["company_key", cfg.date_col], observed=True)
        .agg(
            company_day_amount=("amount", "sum"),
            company_day_abs_amount=("amount_abs", "sum"),
            company_day_row_count=(cfg.id_col, "nunique"),
        )
        .reset_index()
    )
    out = work.merge(company_day, on=["company_key", cfg.date_col], how="left", validate="many_to_one")
    out = out.merge(company_history, on="company_key", how="left", validate="many_to_one")

    cal = pd.DatetimeIndex(calendar)
    out["log_abs_amount"] = np.log1p(out["amount_abs"])
    out["dayofweek"] = out[cfg.date_col].dt.dayofweek
    out["dayofmonth"] = out[cfg.date_col].dt.day
    out["is_weekend"] = out["dayofweek"].ge(5).astype(int)
    out["is_month_start"] = out[cfg.date_col].dt.is_month_start.astype(int)
    out["is_month_end"] = out[cfg.date_col].dt.is_month_end.astype(int)
    out["days_since_start"] = (out[cfg.date_col] - cal.min()).dt.days.astype(float)
    out["days_to_end"] = (cal.max() - out[cfg.date_col]).dt.days.astype(float)
    if "TRADE_DATE" in out.columns:
        out["lead_days"] = (out[cfg.date_col] - out["TRADE_DATE"]).dt.days.astype(float)
        out["known_at_least_d1"] = out["lead_days"].ge(1).astype(float)
        out["known_same_day"] = out["lead_days"].eq(0).astype(float)
    else:
        out["lead_days"] = np.nan
        out["known_at_least_d1"] = np.nan
        out["known_same_day"] = np.nan

    keep_cols = [
        cfg.id_col,
        "company_key",
        cfg.date_col,
        "amount",
        "amount_abs",
        "amount_band",
        "amount_band_order",
        "amount_band_display",
        "log_abs_amount",
        "company_day_amount",
        "company_day_abs_amount",
        "company_day_row_count",
        "dayofweek",
        "dayofmonth",
        "is_weekend",
        "is_month_start",
        "is_month_end",
        "days_since_start",
        "days_to_end",
        "lead_days",
        "known_at_least_d1",
        "known_same_day",
        "company_row_count",
        "company_active_days",
        "company_total_abs_amount",
        "company_active_rate",
        "company_adi",
        "company_cv2",
        "company_sbc_label",
        "company_q90_abs_daily_amount",
        "company_max_abs_daily_amount",
        "company_max_day_abs_share",
    ]
    return out[keep_cols].copy()


def cluster_transaction_regime_features(
    features: pd.DataFrame,
    cluster_counts=AUTO_BAND_CLUSTER_COUNTS,
    cfg: SeriesConfig = cfg,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if features.empty:
        return features.copy(), pd.DataFrame(), pd.DataFrame()

    _, scaled = prepare_cluster_matrix(features, exclude_columns=[cfg.id_col])
    clustered = features.copy()
    clustered["auto_series_id"] = pd.NA
    clustered["cluster_raw"] = pd.NA
    clustered["cluster_kind"] = "clustered"
    band_values = sorted(
        clustered["amount_band"].dropna().unique(),
        key=lambda value: int(clustered.loc[clustered["amount_band"].eq(value), "amount_band_order"].iloc[0]),
    )

    for band in band_values:
        band_idx = clustered.index[clustered["amount_band"].eq(band)].tolist()
        if not band_idx:
            continue
        k = min(max(1, int(cluster_counts.get(str(band), 1))), len(band_idx))
        if k == 1:
            raw_labels = np.zeros(len(band_idx), dtype=int)
        else:
            raw_labels = KMeans(n_clusters=k, random_state=0, n_init=50).fit_predict(scaled.loc[band_idx])
        raw_series = pd.Series(raw_labels, index=band_idx, name="cluster_raw")
        order_frame = clustered.loc[band_idx, ["amount_abs"]].assign(cluster_raw=raw_series)
        cluster_order = order_frame.groupby("cluster_raw", observed=True)["amount_abs"].sum().sort_values(ascending=False).index.tolist()
        label_map = {raw: f"TX_{str(band).upper()}_{i + 1:02d}" for i, raw in enumerate(cluster_order)}
        clustered.loc[band_idx, "cluster_raw"] = [f"{band}_{raw}" for raw in raw_labels]
        clustered.loc[band_idx, "auto_series_id"] = [label_map[raw] for raw in raw_labels]

    if clustered["auto_series_id"].isna().any():
        raise ValueError("Some transaction-regime rows were not assigned")

    clustered = add_pca_coordinates(clustered, scaled)
    profile = scaled.assign(auto_series_id=clustered["auto_series_id"].astype(str).to_numpy()).groupby("auto_series_id", observed=True).median().reset_index()
    return clustered, scaled, profile


transaction_regime_features = build_transaction_feature_frame(df_trf_after_manual_banded, calendar)
transaction_regime_clusters, transaction_regime_feature_matrix_scaled, transaction_regime_cluster_profile_scaled = cluster_transaction_regime_features(transaction_regime_features)

if transaction_regime_clusters.empty:
    transaction_regime_membership = pd.DataFrame(columns=[cfg.id_col, "series_id", "company_key", "amount_band", "amount_band_order", cfg.date_col, "amount", "amount_abs", "cluster_kind"])
else:
    transaction_regime_membership = transaction_regime_clusters.rename(columns={"auto_series_id": "series_id"}).copy()
    transaction_regime_membership = transaction_regime_membership[
        [cfg.id_col, "series_id", "company_key", "amount_band", "amount_band_order", cfg.date_col, "amount", "amount_abs", "cluster_kind"]
    ]

validate_strategy_membership(transaction_regime_membership, df_trf_after_manual_banded, "transaction_regime")
transaction_regime_series_daily = build_dense_daily_series(
    transaction_regime_membership,
    calendar,
    date_col=cfg.date_col,
    amount_col="amount",
    series_col="series_id",
    series_kind="transaction_regime",
)
transaction_regime_band_daily = build_band_total_daily_series(
    transaction_regime_membership,
    calendar,
    prefix="TX_BAND",
    series_kind="transaction_regime_band_total",
)
transaction_regime_all_daily = pd.concat([transaction_regime_series_daily, transaction_regime_band_daily], ignore_index=True, sort=False)

validate_dense_daily(transaction_regime_series_daily, calendar, "transaction_regime_series_daily")
validate_dense_daily(transaction_regime_band_daily, calendar, "transaction_regime_band_daily")

transaction_regime_series_summary = build_series_summary(transaction_regime_membership, transaction_regime_series_daily, "transaction_regime")
transaction_regime_band_summary = build_series_summary(
    membership_view_for_band_totals(transaction_regime_membership, "TX_BAND"),
    transaction_regime_band_daily,
    "transaction_regime_band_total",
)
transaction_regime_series_company_detail = build_series_company_detail(transaction_regime_membership)
transaction_regime_company_series_detail = transaction_regime_series_company_detail.sort_values(
    ["company_key", "amount_band_order", "series_id"], ascending=[True, True, True]
).reset_index(drop=True)
transaction_regime_band_company_detail = build_series_company_detail(membership_view_for_band_totals(transaction_regime_membership, "TX_BAND"))

print("Transaction regime rows:", len(transaction_regime_membership))
print("Transaction regime companies:", transaction_regime_membership["company_key"].nunique() if not transaction_regime_membership.empty else 0)
print("Transaction regime generated series:", transaction_regime_series_summary["series_id"].nunique() if not transaction_regime_series_summary.empty else 0)
print("Transaction regime band-total series:", transaction_regime_band_daily["series_id"].nunique() if not transaction_regime_band_daily.empty else 0)


In [ ]:

display(transaction_regime_series_summary)
display(transaction_regime_band_summary)
display(transaction_regime_series_company_detail)
display(transaction_regime_company_series_detail)
display(transaction_regime_band_company_detail)
display(transaction_regime_clusters.head(30))
display(transaction_regime_all_daily.head())


In [ ]:

plot_regime_daily(transaction_regime_series_daily, transaction_regime_series_summary, "Transaction regime generated daily series by amount band")
plot_band_totals(transaction_regime_band_daily, "Transaction regime band-total daily series")
plot_cluster_map(transaction_regime_clusters, "Transaction regime clustering map")
plot_volume_bar(transaction_regime_series_summary, "Transaction regime series volume and concentration")
plot_cluster_heatmap(transaction_regime_cluster_profile_scaled, "Transaction regime cluster profile heatmap")

if PLOT_SERIES_IDS:
    plot_series = transaction_regime_all_daily.loc[transaction_regime_all_daily["series_id"].isin(PLOT_SERIES_IDS)].copy()
    if not plot_series.empty:
        fig = px.line(
            plot_series,
            x=cfg.date_col,
            y="amount",
            color="series_id",
            labels={cfg.date_col: "Value date", "amount": "Daily amount", "series_id": "Series"},
            title="Selected transaction-regime series",
        )
        fig.update_layout(template="plotly_white", height=430, legend_title_text="")
        fig.show()
